## Reading in and combining files

In [ ]:
import pandas as pd
import numpy as np

### Read in both files

In [2]:
columbia_df = pd.read_csv('Columbia_filtered.csv')
hcmst_df = pd.read_csv('hCMST_Wave1_Features.csv')

### Columns in Both Dataset

In [3]:
target_columns_dict = [
    "Subject age",
    "Partner age",
    "Subject race",
    "Partner race",
    "Subject gender",
    "Partner gender",
    #"Attraction",
    'Interracial relationship' 
]

In [4]:
columbia_df = columbia_df[target_columns_dict]  
hcmst_df = hcmst_df[target_columns_dict]

In [5]:
print(columbia_df.shape)
columbia_df.head()

(7994, 7)


,Subject age,Partner age,Subject race,Partner race,Subject gender,Partner gender,Interracial relationship
0,21,27,ASIAN,WHITE,0,1,1
1,21,22,ASIAN,WHITE,0,1,1
2,21,22,ASIAN,ASIAN,0,1,0
3,21,23,ASIAN,WHITE,0,1,1
4,21,24,ASIAN,HISPANIC,0,1,1


In [6]:
print(hcmst_df.shape)
hcmst_df.head()

(3385, 7)


,Subject age,Partner age,Subject race,Partner race,Subject gender,Partner gender,Interracial relationship
0,1,41,NATIVE AMERICAN,WHITE,1,0,1
1,1,71,WHITE,WHITE,1,0,0
2,1,41,WHITE,WHITE,0,1,0
3,1,51,WHITE,ASIAN,0,1,1
4,1,31,WHITE,WHITE,0,1,0


In [7]:
binary_cols = [ 
    "Subject gender", 
    "Partner gender", 
    'Interracial relationship'
]

# Need to One-Hot Encoding)
nominal_cols = ['Subject race', 
                'Partner race', 
                ]

# leave as numeric, scale if needed
ordinal_cols = [ ]

# leave as numeric, scale if needed
continuous_cols = [
    'Subject age', 
    'Partner age'
]

## COMBINE DATASET

In [8]:
# ADD A COLUMN SO WE KNOW WHICH ROW CAME FROM WHICH DATASET
columbia_df['Dataset_Source'] = 'Columbia'
hcmst_df['Dataset_Source'] = 'hCMST'

# 2. Combine them
combined_df = pd.concat([columbia_df, hcmst_df], ignore_index=True)

In [9]:
combined_df.columns

Index(['Subject age', 'Partner age', 'Subject race', 'Partner race',
       'Subject gender', 'Partner gender', 'Interracial relationship',
       'Dataset_Source'],
      dtype='object')

## Prepare

### Drop Rows with Missing values

In [10]:
print(combined_df.isna().sum())
combined_df = combined_df.dropna().reset_index(drop=True)


Subject age                  0
Partner age                  0
Subject race                15
Partner race                14
Subject gender               0
Partner gender               0
Interracial relationship     0
Dataset_Source               0
dtype: int64


### Encode Catagorical Variables

In [11]:
combined_df = pd.get_dummies(combined_df, columns=nominal_cols, dtype=int)

### Standardize Age

In [12]:
from sklearn.preprocessing import StandardScaler

for col in continuous_cols:
    combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')

combined_df = combined_df.dropna(subset=continuous_cols).reset_index(drop=True)

scaler = StandardScaler()

# continuous_cols = ['Subject age', 'Partner age']
combined_df[continuous_cols] = scaler.fit_transform(combined_df[continuous_cols])

In [15]:
print(combined_df.shape)
combined_df.head()

(11331, 20)


,Subject age,Partner age,Subject gender,Partner gender,Interracial relationship,Dataset_Source,Subject race_ASIAN,Subject race_BLACK,Subject race_HISPANIC,Subject race_LATINO,Subject race_NATIVE AMERICAN,Subject race_OTHER,Subject race_WHITE,Partner race_ASIAN,Partner race_BLACK,Partner race_HISPANIC,Partner race_LATINO,Partner race_NATIVE AMERICAN,Partner race_OTHER,Partner race_WHITE
0,0.176138,-0.363906,0,1,1,Columbia,1,0,0,0,0,0,0,0,0,0,0,0,0,1
1,0.176138,-0.761820,0,1,1,Columbia,1,0,0,0,0,0,0,0,0,0,0,0,0,1
2,0.176138,-0.761820,0,1,0,Columbia,1,0,0,0,0,0,0,1,0,0,0,0,0,0
3,0.176138,-0.682237,0,1,1,Columbia,1,0,0,0,0,0,0,0,0,0,0,0,0,1
4,0.176138,-0.602654,0,1,1,Columbia,1,0,0,0,0,0,0,0,0,1,0,0,0,0


In [14]:
combined_df.to_csv('combined_dataset_matching_features.csv', index=False)